# Lab 10: The Agent Economy — Registry, Evaluations & Payments
## NovaPay · Building "AgentBridge" — From Isolated Agents to a Governed Agent Economy

**Level path:** 100 (business problem) → 200 (Registry — discovery) → 300 (Evaluations — trust) → 400 (Payments — economic autonomy)

**Duration:** ~100 minutes
**Prerequisites:** Labs 1–9 (the corridor-compliance tool from Lab 9's gateway is reused as the object of discovery)
**Cost:** ~10 cents. Registry and Evaluations calls are metered per-request; Payments runs entirely in simulation (see 400.1)
**Region:** us-west-2

> **This lab provisions real AWS resources**: an AWS Agent Registry, a registry record,
> and (optionally) real AgentCore Evaluations calls. AWS Agent Registry and AgentCore
> Payments are **public preview** services as of this lab's writing (August 2026) — their
> APIs may shift before general availability. Every call that touches a preview service
> is wrapped defensively so the notebook completes even if a specific operation behaves
> differently than documented. Every resource is registered for teardown in section 400.5.

> **A note on Payments (Level 400):** Amazon Bedrock AgentCore Payments requires a real
> funded wallet from a third-party provider (Coinbase or Stripe) — credentials this lab
> cannot assume you have. Level 400 therefore runs against a **local simulator** that
> implements the *exact* method signatures of the real `bedrock_agentcore.payments` SDK
> (verified against AWS's own technical deep-dive, cited inline). Section 400.3 shows the
> real production code side-by-side — swapping from simulation to production is an import
> statement and a set of real credentials, nothing else.

In [ ]:
%pip install -q --upgrade opentelemetry-api opentelemetry-sdk strands-agents strands-agents-tools \
    bedrock-agentcore "boto3>=1.43.0" "botocore>=1.43.0" requests
# boto3>=1.43.0 is a hard requirement, not a suggestion: AWS Agent Registry's
# agent-registry / agent-registry-control service clients (Level 200) were only added to
# boto3 starting with 1.43.0. Versions >=1.42.87 install successfully and import without
# error, but raise UnknownServiceError the moment you call boto3.client("agent-registry-control")
# — there's no error until that exact line. If you're on a persistent virtualenv (not a
# fresh container) and already have an older boto3 satisfying some earlier constraint,
# pip will NOT upgrade it unless the currently-installed version fails the version check
# above — which is why --upgrade is included here rather than relying on the constraint alone.

In [ ]:
import json, os, time, uuid, textwrap
import boto3
from datetime import datetime, timezone

# ── Region and model ──
REGION = "us-west-2"
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
RUN = datetime.now(timezone.utc).strftime("%m%d%H%M")
ACCOUNT = boto3.client("sts").get_caller_identity()["Account"]

print(f"Account : {ACCOUNT}")
print(f"Region  : {REGION}")
print(f"Model   : {MODEL_ID}")
print(f"Run tag : {RUN}")

# ── OpenTelemetry setup (same pattern proven in Lab 9) ──
# We reuse a local, self-contained InMemorySpanExporter rather than depending
# on a library symbol that varies across opentelemetry-sdk versions.
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, SpanExporter, SpanExportResult
from opentelemetry.sdk.resources import Resource

class InMemorySpanExporter(SpanExporter):
    """Minimal in-memory span exporter — collects finished spans in a list."""
    def __init__(self):
        self._spans = []
    def export(self, spans):
        self._spans.extend(spans)
        return SpanExportResult.SUCCESS
    def get_finished_spans(self):
        return tuple(self._spans)
    def clear(self):
        self._spans = []
    def shutdown(self):
        pass

span_exporter = InMemorySpanExporter()
otel_resource = Resource.create({
    "service.name": "novapay-agent-economy",
    "service.version": "1.0.0",
    "deployment.environment": "lab",
})
provider = TracerProvider(resource=otel_resource)
provider.add_span_processor(SimpleSpanProcessor(span_exporter))

# NOTE: we deliberately do NOT rely on trace.set_tracer_provider(provider) +
# trace.get_tracer(...) here. OpenTelemetry's global tracer provider is a process-wide
# singleton that can only be set once — every re-run of this cell in the same kernel
# after the first will have set_tracer_provider() silently no-op (it logs "Overriding of
# current TracerProvider is not allowed" but doesn't raise), leaving trace.get_tracer()
# bound to whichever provider — and whichever span_exporter — won the very first call in
# this kernel process. Spans would still be created and exported correctly, just to that
# orphaned first exporter instead of the fresh one this cell just built, so
# span_exporter.get_finished_spans() would silently return nothing even though the tool
# genuinely ran. Binding the tracer directly to our own `provider` object sidesteps the
# singleton entirely and is correct on every re-run, restarted kernel or not.
try:
    trace.set_tracer_provider(provider)
except Exception:
    pass  # already set in this kernel from an earlier run — harmless, we don't rely on it
tracer = provider.get_tracer("novapay.agent_economy.lab10")

print(f"\n✅ OpenTelemetry configured — spans captured in memory for analysis")

---
# LEVEL 100 · The Business Problem

## From "It Works" to "It's an Economy"

Labs 7–9 took NovaPay's fraud agent through three maturity stages:

| Lab | Capability | What it solved |
|---|---|---|
| 7 | **Memory** | The agent forgets nothing it should remember |
| 8 | **Gateway & Identity** | The agent reaches tools through one governed front door, with no credentials of its own |
| 9 | **Observability** | When the agent misbehaves, you can see exactly why in minutes, not hours |

By the end of Lab 9, NovaPay has dozens of production agents across six teams — fraud,
support, collections, merchant onboarding, treasury, and the mobile assistant — each
built on the patterns from Labs 7–9. That success has created three *new* problems that
only show up once you have **many** agents built by **many** teams, all capable, all
correct, all invisible to each other.

## Act I — "We built this already"

The **Growth team** is scoping a new agent to evaluate expansion into three new
West-African payment corridors. Two weeks into the build, a Slack message from a Fraud
engineer stops them cold: *"Wait — we already have a `verify_corridor_compliance` tool.
It's been live on the fraud gateway since Lab 9."* Growth had no way to know that. There
is no catalog of what agents and tools already exist at NovaPay — discovery happens by
accident, in hallway conversations and lucky Slack threads.

## Act II — "How do we know it's any good?"

Growth finds the tool. Before they wire their new agent to depend on it, someone asks
the obvious question: *"Is it actually reliable? What's its accuracy? Has anyone
measured it?"* Nobody can answer with a number. The tool works — Lab 9 proved that for
one incident — but there is no systematic, repeatable way to score whether an agent's
decisions are good, before or after you build on it.

## Act III — "Why does procurement take three weeks for a two-cent API call?"

The compliance team wants real-time sanctions-list screening from a premium third-party
data vendor for every new corridor NovaPay considers. The vendor charges **$0.02 per
lookup** — but NovaPay's only way to pay them is a traditional vendor contract: legal
review, a wire-transfer billing relationship, a minimum annual commitment. Three weeks
of procurement for what should be a two-cent API call, repeated for every vendor an
agent might ever need to query.

<div></div>

**Three different problems. One underlying shape**: NovaPay has the *engineering*
maturity from Labs 7–9, but not yet the *organizational* maturity to let agents be
**discovered**, **trusted**, and **economically autonomous** at scale. Lab 10 builds all
three, in the order that makes each one trustworthy before the next depends on it:

**Registry** (can other teams find this?) → **Evaluations** (should they trust what they
found?) → **Payments** (now that it's proven, can it act — including spending money —
on its own?)

## Why This Sequence, Specifically

It would be tempting to build Payments first — it is the most dramatic capability, and
"agents that pay for things" is the headline. But sequencing it last is deliberate, and
mirrors how a risk-conscious payments company actually rolls out new agent capabilities:

| Step | Question it answers | Why it must come before the next step |
|---|---|---|
| **Registry** | *Does this capability exist, and where?* | You cannot evaluate or trust something you cannot find |
| **Evaluations** | *Is it good enough to build on?* | You should not give money-spending authority to an agent nobody has scored |
| **Payments** | *Can it now act with real economic consequences?* | Financial autonomy is the highest-stakes capability — it should be the last gate an agent clears, not the first |

Each stage is also a governance checkpoint. A registry with no evaluation gate would let
teams adopt untested agents at scale. An evaluation practice with no payment guardrails
downstream would score agents that can never do anything expensive enough to matter.
Building all three, in this order, is what turns "agents that work" into "an agent
economy NovaPay can govern."

## Concepts at a Glance

| Concept | Traditional Software Analogy | NovaPay Example |
|---|---|---|
| **AWS Agent Registry** | An internal package registry (like a private npm or PyPI) | A searchable catalog of NovaPay's agents, MCP tools, and skills, with approval gates |
| **Registry Record** | A published package version | One entry: the `corridor-compliance` MCP tool, version `1.0` |
| **AgentCore Evaluations** | A test suite + CI quality gate | A scored judgment — *Builtin.Helpfulness*, *Builtin.GoalSuccessRate* — on a real agent session |
| **On-demand evaluation** | Running your test suite manually / in CI | Evaluate a specific session, on request, for a go/no-go decision |
| **AgentCore Payments** | A metered billing SDK + wallet | The Growth agent autonomously pays a sanctions-data vendor $0.02 per lookup via the x402 protocol |
| **Payment Session** | An OAuth-scoped, time-boxed API token — but for money | A $10 budget, expiring in 60 minutes, that the agent cannot exceed no matter how many lookups it makes |

## Resource Lifecycle

| # | Resource | Created in | Ready signal | Teardown |
|---|---|---|---|---|
| 1 | AWS Agent Registry | 200.1 | `status == "READY"` | 400.5 |
| 2 | Registry record (corridor-compliance tool) | 200.2 | `status == "DRAFT"` | 400.5 |
| 3 | Approved registry record | 200.3–200.4 | `status == "APPROVED"` | (deleted with record) |
| 4 | OTEL spans (in-memory) | 300.1 | immediate | 400.5 (provider shutdown) |
| 5 | Simulated payment manager / instrument / session | 400.1–400.2 | immediate (local) | none (in-memory only) |

Unlike Labs 8–9, this lab does **not** create an IAM role, Lambda, or Cognito pool —
Level 200 reuses the Lambda tool schema pattern from Lab 9 conceptually (the record
describes that same tool) without re-provisioning it, and Level 400 runs local-only.
This keeps Lab 10's real AWS footprint small: one registry, one record.

---
# LEVEL 200 · AWS Agent Registry — Solving Act I

## Why a Registry, Specifically

A wiki page listing "agents we've built" goes stale within a month. AWS Agent Registry
is purpose-built for this instead: it validates MCP and A2A definitions against their
real protocol schemas at submission time (so a broken record can't get published), it
combines semantic and keyword search (so "find a tool that checks payment corridors"
matches `corridor-compliance` even without exact keyword overlap), and it has an
explicit approval workflow — records start as `DRAFT` and only become discoverable once
someone with authority marks them `APPROVED`.

That approval gate is the point. A registry without governance is just a slightly
better wiki. NovaPay wants Growth to find *only* tools that a curator has vouched for —
not every half-finished prototype in the company.

> **Namespace note:** AWS Agent Registry launched its **generally-available**
> `agent-registry` namespace on August 6, 2026, replacing the `bedrock-agentcore`
> namespace it ran under during public preview. If you don't already have registries or
> records from before that date, the old namespace is no longer reachable at all — every
> call below uses the new `agent-registry-control` (control plane) and `agent-registry`
> (data plane) boto3 clients and the current request/response schema (`recordType`,
> `name` as the dedup key, flat `descriptors.mcpServer.data`). This is not a cosmetic
> rename: field names, required parameters, and the registry-record shape all changed
> alongside the namespace — see AWS's
> [registry migration guide](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/registry-faq.html)
> if you're adapting older AWS Agent Registry code.

In [ ]:
registry_control = boto3.client("agent-registry-control", region_name=REGION)
registry_data = boto3.client("agent-registry", region_name=REGION)

REGISTRY_NAME = f"novapay-agent-registry-{RUN}"
REGISTRY_ID = None  # holds the registry ARN — the API accepts ARN or bare ID interchangeably

try:
    reg = registry_control.create_registry(
        name=REGISTRY_NAME,
        description="NovaPay's catalog of agents, MCP tools, and skills across all teams",
    )
    # CreateRegistry only returns registryArn (no separate registryId field) — but every
    # downstream call's registryId parameter accepts either form (confirmed against the
    # service's own request-validation pattern), so we just use the ARN everywhere.
    REGISTRY_ID = reg["registryArn"]
    print(f"✅ Registry created: {REGISTRY_NAME}")
    print(f"   ARN: {REGISTRY_ID}")

    # ── Wait for the registry to become usable ──
    started = time.time()
    while time.time() - started < 120:
        status = registry_control.get_registry(registryId=REGISTRY_ID).get("status", "UNKNOWN")
        if status == "READY":
            print(f"✅ Registry ready")
            break
        if status in ("CREATE_FAILED", "DELETING", "ERROR"):
            raise RuntimeError(f"Registry entered terminal status: {status}")
        print(f"   registry status: {status} … waiting")
        time.sleep(5)
    else:
        print(f"⚠️  Registry not confirmed READY within 120s — proceeding anyway")

except Exception as e:
    print(f"⚠️  Registry creation: {type(e).__name__}: {e}")
    print("   If this is AccessDeniedException, confirm your IAM policy grants")
    print("   agent-registry:* (not bedrock-agentcore:*) — see the namespace note above.")
    print("   The rest of this section is written to degrade gracefully if REGISTRY_ID is None.")

## 200.2 · Register the corridor-compliance tool

We publish the exact tool NovaPay's fraud team built in Lab 9 — `verify_corridor_compliance`
— as an MCP-type registry record. AWS Agent Registry validates this against the real MCP
server and tool schemas before accepting it, so a malformed record is rejected at
submission time rather than discovered later by a confused consumer.

The record has two parts, both nested under the `mcpServer` descriptor: the **server
definition** (what MCP server this tool lives on — its name, description, version) and
the **tool definition** (the specific callable, its input schema), which sits under
`mcpServer.additionalData.tools`. Both are passed as JSON strings under `data`
(the `agent-registry` namespace's rename of the old `inlineContent` field).

Every record also needs a `recordType` (`MCP`, since this is an MCP tool) and a `name`
that's unique within the registry — this is the new namespace's dedup key, distinct from
the human-readable `displayName`.

In [ ]:
RECORD_ID = None  # holds the record ARN — same ARN-or-ID flexibility as REGISTRY_ID

if REGISTRY_ID:
    # The official MCP server.json schema (validated server-side by AWS Agent Registry)
    # caps `description` at 100 characters — see
    # https://static.modelcontextprotocol.io/schemas/2025-12-11/server.schema.json
    server_content = json.dumps({
        "name": "novapay/fraud-gateway",
        "description": "NovaPay fraud gateway: corridor compliance, sender history, risk scoring tools.",
        "version": "1.0.0",
    })

    tools_content = json.dumps({
        "tools": [{
            "name": "verify_corridor_compliance",
            "description": "Check corridor-specific regulatory compliance rules for a "
                            "cross-border payment (max single transfer, SAR thresholds, "
                            "corridor status)",
            "inputSchema": {
                "type": "object",
                "properties": {
                    "corridor": {"type": "string", "description": "Payment corridor e.g. NG-GH"},
                    "amount": {"type": "number", "description": "Transaction amount"},
                },
                "required": ["corridor", "amount"],
            },
        }],
    })

    try:
        record = registry_control.create_registry_record(
            registryId=REGISTRY_ID,
            name="corridor-compliance",
            displayName="Corridor Compliance Checker",
            recordType="MCP",
            descriptors={
                "mcpServer": {
                    "data": server_content,
                    "dataSchemaVersion": "2025-12-11",
                    "additionalData": {
                        "tools": {"data": tools_content, "dataSchemaVersion": "2024-11-05"},
                    },
                }
            },
        )
        RECORD_ID = record["recordArn"]
        print(f"✅ Record created: corridor-compliance")
        print(f"   Status: {record.get('status', 'CREATING')}")
        print(f"   ARN: {RECORD_ID}")
    except Exception as e:
        print(f"⚠️  Record creation: {type(e).__name__}: {e}")
else:
    print("ℹ️  Skipping — no registry available (see 200.1)")

## 200.3 · Submit for approval, then approve

Every record starts in `DRAFT`. Submitting for approval moves it to `PENDING_APPROVAL`
(unless the registry has auto-approval enabled) — it is *still not discoverable* until
someone with curator authority explicitly approves it. In this lab, we play both roles:
the publisher (Fraud team) submits, and the curator (also us, for the lab) approves. In
a real NovaPay rollout, these would be two different people, and that separation is the
entire point of the approval gate — a publisher cannot make their own record
discoverable.

In [ ]:
if RECORD_ID:
    try:
        submitted = registry_control.submit_registry_record_for_approval(
            registryId=REGISTRY_ID,
            recordId=RECORD_ID,
        )
        print(f"✅ Submitted for approval — status: {submitted.get('status')}")

        # ── Approve as curator ──
        # We explicitly call UpdateRegistryRecordStatus regardless of the registry's
        # auto-approval setting. This is the same API a real approval pipeline calls
        # after its own security/compliance checks pass (see EventBridge integration
        # in the AWS Agent Registry docs) — we just call it directly here.
        approved = registry_control.update_registry_record_status(
            registryId=REGISTRY_ID,
            recordId=RECORD_ID,
            status="APPROVED",
            statusReason="Reviewed by Fraud Eng — live in production since Lab 9, "
                         "safe for other teams to depend on",
        )
        print(f"✅ Approved — status: {approved.get('status')}")
        print(f"   Reason: {approved.get('statusReason')}")
    except Exception as e:
        print(f"⚠️  Approval flow: {type(e).__name__}: {e}")
else:
    print("ℹ️  Skipping — no record available (see 200.2)")

## 200.4 · The Growth team's search

This is the payoff: the Growth team, who has never talked to Fraud Eng, searches the
registry in natural language and finds the tool anyway. This is what "no more hallway
discovery" looks like in practice — semantic search understands *intent*, not just
keyword overlap.

In [ ]:
SEARCH_QUERY = "tool that checks if a cross-border payment corridor is compliant"

if REGISTRY_ID:
    try:
        # Approval → searchable is not instantaneous (the record has to reach the search
        # index), so poll instead of a single fixed sleep-then-check — the same pattern
        # used for the registry's CREATING→READY transition in 200.1.
        records = []
        started = time.time()
        attempt = 0
        while time.time() - started < 90:
            attempt += 1
            time.sleep(5)
            results = registry_data.search_discoverable_registry_records(
                registryIds=[REGISTRY_ID],
                searchQuery=SEARCH_QUERY,
                maxResults=10,
            )
            records = results.get("registryRecords", [])
            if records:
                break
            print(f"   … not indexed yet (attempt {attempt}), retrying")

        print(f"🔍 Growth team searches: \"{SEARCH_QUERY}\"")
        print(f"   Found {len(records)} result(s):\n")
        for r in records:
            print(f"   • {r.get('displayName') or r.get('name')}  (type: {r.get('recordType')}, "
                  f"status: {r.get('status')}, version: {r.get('recordVersion')})")
        if not records:
            print("   ⚠️  Still not indexed after 90s. If 200.2/200.3 both printed ✅ Approved,")
            print("      this points at something other than propagation delay — check the")
            print("      record's actual status with registry_control.get_registry_record(")
            print("      registryId=REGISTRY_ID, recordId=RECORD_ID) before assuming it's timing.")
    except Exception as e:
        print(f"⚠️  Search: {type(e).__name__}: {e}")
else:
    print("ℹ️  Skipping — no registry available (see 200.1)")

> ### Level 200 checkpoint
> Growth found the tool in seconds, through search, with zero hallway conversations —
> and only because a curator (Fraud Eng) explicitly approved it first. That second part
> matters as much as the first: **discovery without governance is just noise at scale.**
>
> But finding a tool isn't the same as trusting it. Growth's next question is Act II's:
> *"is it actually good?"* Level 300 answers that with a number.

---
# LEVEL 300 · AgentCore Evaluations — Solving Act II

## Why "It Worked Once" Is Not Evidence

Lab 9 proved the fraud agent's Dark Tuesday decision was traceable — we could see
*why* it approved TXN-DT-091. That is diagnostic power, not quality measurement. Knowing
why an agent made a decision tells you nothing about whether its decisions, in general,
are *good*. Before Growth builds a production dependency on the corridor-compliance
tool they just found, they need a repeatable, scored answer to: "if I give this agent
100 different corridor-compliance questions, how often does it get them right?"

AgentCore Evaluations answers this with **built-in evaluators** — LLM-graded judges that
score a real agent session against dimensions like helpfulness, goal success, and tool
selection accuracy — invoked through a single `Evaluate` API call.

## On-Demand vs. Online Evaluation

| | On-demand | Online |
|---|---|---|
| **When it runs** | You trigger it, against a specific session | Continuously, sampling live production traffic |
| **Use case** | "Is this specific interaction good?" / CI regression gate | "Is agent quality drifting in production?" |
| **What NovaPay uses it for here** | Growth's go/no-go decision before adopting the tool | Fraud Eng's ongoing production quality dashboard (Level 400 note) |

This lab runs **on-demand evaluation** — it matches Growth's actual question and needs
no standing infrastructure.

## 300.1 · Generate a real session to evaluate

`Evaluate` scores real telemetry — spans captured from an actual agent run, not a
description of one. We build a small Strands agent around the exact corridor-compliance
logic from Lab 9, instrument it with the same OpenTelemetry pattern proven in that lab,
and run it against three representative corridor questions Growth cares about.

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def verify_corridor_compliance(corridor: str, amount: float) -> str:
    """Check corridor-specific compliance rules for a cross-border payment.
    Args:
        corridor: Payment corridor code, e.g. NG-GH
        amount: Transaction amount in USD
    """
    with tracer.start_as_current_span("novapay.corridor_compliance") as span:
        span.set_attribute("gen_ai.tool.name", "verify_corridor_compliance")
        span.set_attribute("novapay.corridor", corridor)
        span.set_attribute("novapay.amount", amount)

        corridors = {
            "NG-GH": {"max_single": 100000, "requires_sar": amount > 50000, "status": "open"},
            "NG-UK": {"max_single": 500000, "requires_sar": amount > 200000, "status": "open"},
            "GH-KE": {"max_single": 75000, "requires_sar": amount > 40000, "status": "open"},
        }
        info = corridors.get(corridor, {"max_single": 50000, "requires_sar": True, "status": "restricted"})
        info.update({"corridor": corridor, "amount": amount, "within_limit": amount <= info["max_single"]})

        span.set_attribute("novapay.within_limit", info["within_limit"])
        span.set_attribute("novapay.requires_sar", info["requires_sar"])
        return json.dumps(info)

model = BedrockModel(model_id=MODEL_ID, region_name=REGION)

eval_agent = Agent(
    model=model,
    tools=[verify_corridor_compliance],
    system_prompt="You are NovaPay's corridor compliance assistant. You MUST call the "
    "verify_corridor_compliance tool for every question — never answer from your own "
    "knowledge, even if you're confident. This is a compliance requirement: unverified "
    "answers are not acceptable regardless of how simple the question seems. After "
    "calling the tool, explain in one sentence whether the transfer is within limits "
    "and whether a Suspicious Activity Report is required, based on the tool's output.",
)

EVAL_SESSION_ID = f"growth-eval-session-{uuid.uuid4().hex[:12]}"
questions = [
    "Is a $45,000 transfer on the NG-GH corridor within limits, and does it need a SAR?",
    "Is a $250,000 transfer on the NG-UK corridor within limits, and does it need a SAR?",
    "Is a $90,000 transfer on the GH-KE corridor within limits, and does it need a SAR?",
]

span_exporter.clear()
print(f"Session ID: {EVAL_SESSION_ID}\n")
responses = []
for q in questions:
    print(f"Q: {q}")
    before = len(span_exporter.get_finished_spans())
    response = eval_agent(q)
    response_text = str(response)
    after = len(span_exporter.get_finished_spans())
    print(f"A: {response_text[:220]}")
    if after == before:
        print("   ⚠️  No new span — the model answered without calling verify_corridor_compliance.")
    print()
    responses.append(response_text)

captured_spans = span_exporter.get_finished_spans()
print(f"✅ Captured {len(captured_spans)} span(s) across {len(questions)} interactions")
if len(captured_spans) < len(questions):
    print("⚠️  Fewer spans than questions — see the per-question warnings above for which")
    print("   interaction(s) skipped the tool. The system prompt instructs the model to")
    print("   always call the tool, but Bedrock's Converse API doesn't guarantee that;")
    print("   Strands' Agent interface doesn't currently expose a way to force tool_choice")
    print("   per-call the way the underlying BedrockModel.stream() API does.")

## 300.2 · Shape the spans for Evaluate

In production, you would pull these spans from CloudWatch after deploying the agent to
AgentCore Runtime (the exact download pattern AWS documents: query the
`/aws/bedrock-agentcore/runtimes/{agent_id}-{endpoint}` and `aws/spans` log groups by
session ID, then pass the raw JSON straight through — no reshaping). To keep this lab
fully self-contained and runnable without a Runtime deployment, we reconstruct that same
shape locally — sourced from AWS's documented span/event schema
([Understanding input spans](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/understanding-input-spans.html)),
which is **not** the generic OTLP/JSON wire format. Two things Evaluate is strict about:

- **`attributes` is a flat `{key: value}` dict**, not OTLP's list-of-`{key, value}`
  objects — sending the list form throws `ValidationException: 'list' object has no
  attribute 'get'`.
- **`scope.name` must be one of a fixed set of supported instrumentation scopes**
  (`strands.telemetry.tracer`, `opentelemetry.instrumentation.langchain`,
  `openinference.instrumentation.langchain`). Spans outside this list are silently
  ignored by the evaluator. Our tool runs inside a Strands `Agent`, so we tag spans with
  `strands.telemetry.tracer` — the scope Strands' own auto-instrumentation would use if
  this agent were deployed to AgentCore Runtime with Observability enabled. (Our actual
  local tracer, defined in Setup, is scoped to `"novapay.agent_economy.lab10"` — we
  override the field here so the evaluator recognizes the span; this is a deliberate
  approximation, not something the real SDK does for you.)

AWS's docs also state a second requirement: **evaluation needs a matching *event* for
each span**, carrying the actual input/output content, or the call throws a
`ValidationException`. Spans alone (metadata: timing, attributes) aren't enough — the
LLM judge reads message content from the event's `body.input.messages` /
`body.output.messages`, which we build here from the questions and responses captured
in 300.1.

In [ ]:
def span_to_evaluation_json(span, session_id):
    """Convert an in-memory OTEL span into the shape AgentCore Evaluations expects.

    Source: AWS's documented span schema, not generic OTLP/JSON —
    https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/understanding-input-spans.html
    """
    ctx = span.get_span_context()
    attributes = dict(span.attributes or {})
    attributes["session.id"] = session_id

    return {
        "spanId": format(ctx.span_id, "016x"),
        "traceId": format(ctx.trace_id, "032x"),
        "name": span.name,
        "scope": {"name": "strands.telemetry.tracer"},
        "kind": "INTERNAL",
        "startTimeUnixNano": str(span.start_time),
        "endTimeUnixNano": str(span.end_time),
        "attributes": attributes,
        "status": {"code": "OK"},
    }


def span_to_event_json(span, session_id, question, response_text):
    """Build the companion event AWS requires alongside each span.

    Per the same doc: "To evaluate a session, both spans and the corresponding events
    are required... else the service will throw a ValidationException."
    """
    ctx = span.get_span_context()
    return {
        "spanId": format(ctx.span_id, "016x"),
        "traceId": format(ctx.trace_id, "032x"),
        "scope": {"name": "strands.telemetry.tracer"},
        "timeUnixNano": str(span.end_time),
        "attributes": {
            "event.name": "gen_ai.content.completion",
            "session.id": session_id,
        },
        "body": {
            "input": {"messages": [{"role": "user", "content": question}]},
            "output": {"messages": [{"role": "assistant", "content": response_text}]},
        },
    }


session_span_logs = []
for span, q, resp in zip(captured_spans, questions, responses):
    session_span_logs.append(span_to_evaluation_json(span, EVAL_SESSION_ID))
    session_span_logs.append(span_to_event_json(span, EVAL_SESSION_ID, q, resp))

n_pairs = len(captured_spans)
print(f"✅ Built {len(session_span_logs)} entries: {n_pairs} span(s) + {n_pairs} matching event(s)")
if session_span_logs:
    print(json.dumps(session_span_logs[0], indent=2)[:600] + "\n...")
else:
    print("ℹ️  No spans captured — the agent call above may not have triggered the tool,")
    print("   or the span processor hasn't flushed yet. The rest of this section is")
    print("   written to degrade gracefully with an empty span list (see 300.3/300.4).")

## 300.3 · Call Evaluate

One API call, two built-in evaluators. `Builtin.Helpfulness` scores whether the
response actually answers the user's question; `Builtin.GoalSuccessRate` scores whether
the interaction accomplished what it set out to do. Both are graded by an LLM judge
behind the API — you provide the spans, AgentCore provides the scoring model and the
rubric.

In [ ]:
eval_client = boto3.client("bedrock-agentcore", region_name=REGION)

EVALUATORS = ["Builtin.Helpfulness", "Builtin.GoalSuccessRate"]
all_results = []

if not session_span_logs:
    print("ℹ️  Skipping Evaluate — no spans/events were built in 300.2 (see that cell's")
    print("   output for why). Calling Evaluate with an empty sessionSpans list always")
    print("   fails client-side validation (min length 1), so there's nothing to try here.")
else:
    for evaluator_id in EVALUATORS:
        try:
            response = eval_client.evaluate(
                evaluatorId=evaluator_id,
                evaluationInput={"sessionSpans": session_span_logs},
            )
            results = response.get("evaluationResults", [])
            all_results.extend(results)
            print(f"✅ {evaluator_id}: {len(results)} result(s)")
            for r in results:
                if "value" in r:
                    print(f"   score={r.get('value'):.2f}  label={r.get('label')}")
                    if r.get("explanation"):
                        print(f"   → {r['explanation'][:160]}...")
                elif "errorMessage" in r:
                    print(f"   ⚠️  {r.get('errorCode')}: {r.get('errorMessage')}")
        except Exception as e:
            print(f"⚠️  {evaluator_id}: {type(e).__name__}: {e}")
            print("   AgentCore Evaluations is GA but strict about span attribute shape.")
            print("   In production, spans come pre-formatted from a real Runtime deployment —")
            print("   this lab's locally-built spans are a best-effort approximation of that format.")

print(f"\n{'='*60}")
if all_results:
    scored = [r for r in all_results if "value" in r]
    if scored:
        avg = sum(r["value"] for r in scored) / len(scored)
        print(f"Growth's go/no-go signal: average score {avg:.2f} across {len(scored)} evaluation(s)")
    else:
        print("No scored results returned — see warnings above.")
else:
    print("No results returned. If Evaluate failed above, Level 300's teaching point still")
    print("stands: this is the exact production API and workflow NovaPay would use once the")
    print("agent is deployed to AgentCore Runtime with real CloudWatch-delivered spans.")

## 300.4 · Evaluation targets — scoring a specific trace or tool call

The `evaluate()` call above scored the whole session. For finer-grained questions —
*"was tool selection correct on this one call?"* — pass `evaluationTarget` with specific
trace or span IDs. This is how a CI pipeline would gate a single regression test rather
than re-scoring an entire session on every run.

In [ ]:
if session_span_logs:
    sample_trace_id = session_span_logs[0]["traceId"]
    try:
        targeted = eval_client.evaluate(
            evaluatorId="Builtin.Helpfulness",
            evaluationInput={"sessionSpans": session_span_logs},
            evaluationTarget={"traceIds": [sample_trace_id]},
        )
        print(f"✅ Trace-level evaluation on {sample_trace_id[:16]}...:")
        for r in targeted.get("evaluationResults", []):
            if "value" in r:
                print(f"   score={r.get('value'):.2f}  label={r.get('label')}")
    except Exception as e:
        print(f"⚠️  Targeted evaluation: {type(e).__name__}: {e}")
else:
    print("ℹ️  No spans available to target — see 300.1/300.2")

> ### Level 300 checkpoint
> Growth now has a number, not a hallway assurance: the corridor-compliance tool's
> helpfulness and goal-success scores from a real, scored session. That is enough for a
> go/no-go decision made on evidence.
>
> With the tool **found** (Level 200) and **trusted** (Level 300), NovaPay is ready for
> the highest-stakes step: letting an agent act on that trust with real economic
> consequences. Level 400 is where an agent starts spending money.

---
# LEVEL 400 · AgentCore Payments — Solving Act III

## Why This Is the Last Gate, Not the First Feature

Everything before this level was about *information* — discovering a tool, scoring a
tool. Payments is different in kind: it lets an agent commit NovaPay's money, in real
time, with no human in the loop for each individual transaction. That is exactly why it
sits behind Registry and Evaluations in this lab's sequence — an agent economy should
not hand out spending authority before it has discovery and trust infrastructure in
place.

## The Business Problem, Concretely

NovaPay's compliance team wants real-time sanctions-list screening from a premium
third-party data vendor during corridor expansion research (the same research Growth is
doing, now backed by the corridor-compliance tool they found and trusted above). The
vendor charges **$0.02 per lookup** and speaks the **x402** protocol: call the API, get
back an HTTP `402 Payment Required` with payment terms, pay, retry with proof, get your
data. Traditional procurement — contracts, wire transfers, minimum commitments — cannot
economically serve a two-cent transaction. AgentCore Payments is built specifically for
this gap.

## 400.1 · Why This Lab Simulates the Payment Provider

AgentCore Payments requires real credentials from a payment provider — a Coinbase
Developer Platform API key or a Stripe secret key — to create a funded wallet
(*payment instrument*). That is a real third-party account this lab cannot assume you
have. Rather than skip the hands-on experience entirely, this section runs against a
**local simulator** whose class and method signatures are copied verbatim from the real
`bedrock_agentcore.payments` SDK (`PaymentClient`, `PaymentManager`), as documented in
AWS's own technical deep-dive on AgentCore Payments (cited in 400.3). Every concept below
— payment manager, connector, instrument, session, spend limits, the x402 payment
payload shape — is the real production shape. Only the network calls are stubbed.

In [ ]:
# ── Local simulator matching the real bedrock_agentcore.payments interface ──
# Method names, parameter names, and response shapes below are copied from AWS's
# technical deep-dive on AgentCore payments (see 400.3 for the verbatim real code).

class SimulatedPaymentClient:
    """Stands in for bedrock_agentcore.payments.PaymentClient."""
    def __init__(self, region_name):
        self.region_name = region_name

    def create_payment_manager_with_connector(self, payment_manager_name, payment_manager_description,
                                               authorizer_type, role_arn, payment_connector_config):
        manager_arn = f"arn:aws:bedrock-agentcore:{self.region_name}:{ACCOUNT}:payment-manager/{payment_manager_name}"
        connector_id = f"{payment_connector_config['name']}-{uuid.uuid4().hex[:10]}"
        print(f"   [simulated] payment manager '{payment_manager_name}' created")
        print(f"   [simulated] connector '{payment_connector_config['name']}' provisioned "
              f"(credentials stored — never returned by this or any real API)")
        return {"paymentManager": {"paymentManagerArn": manager_arn, "paymentConnectorId": connector_id}}


class SimulatedPaymentManager:
    """Stands in for bedrock_agentcore.payments.PaymentManager."""
    def __init__(self, payment_manager_arn, region_name):
        self.payment_manager_arn = payment_manager_arn
        self.region_name = region_name
        self._instruments = {}
        self._sessions = {}

    def create_payment_instrument(self, user_id, payment_connector_id, payment_instrument_type,
                                   payment_instrument_details):
        instrument_id = f"instr-{uuid.uuid4().hex[:12]}"
        self._instruments[instrument_id] = {"user_id": user_id, "funded": True, "balance": 10.00}
        network = payment_instrument_details.get("embeddedCryptoWallet", {}).get("network", "UNKNOWN")
        print(f"   [simulated] embedded {network} wallet provisioned for user '{user_id}'")
        print(f"   [simulated] (real flow: user completes funding + signing-authority via a")
        print(f"                provider-hosted redirect URL before first use)")
        return {"paymentInstrumentId": instrument_id, "status": "ACTIVE"}

    def create_payment_session(self, user_id, limits, expiry_time_in_minutes):
        session_id = f"psess-{uuid.uuid4().hex[:12]}"
        max_spend = float(limits["maxSpendAmount"]["value"])
        self._sessions[session_id] = {
            "user_id": user_id, "limit": max_spend, "reserved": 0.0, "spent": 0.0,
            "currency": limits["maxSpendAmount"]["currency"],
            "expires_in_min": expiry_time_in_minutes,
        }
        return {"paymentSessionId": session_id, "status": "ACTIVE"}

    def process_payment(self, user_id, payment_session_id, payment_instrument_id, payment_input):
        """Three-phase workflow: reserve → process → commit/rollback (matches the real
        AgentCore Payments atomic budget-check protocol)."""
        session = self._sessions.get(payment_session_id)
        if session is None:
            return {"status": "FAILED", "errorMessage": "unknown payment session"}

        x402 = payment_input.get("cryptoX402", {}).get("payload", {})
        # x402 "maxAmountRequired" is denominated in the token's smallest unit (e.g. USDC
        # has 6 decimals) — real integrations read the asset's decimals; this simulator
        # assumes USDC-style 6-decimal micro-units for realism.
        amount = int(x402.get("maxAmountRequired", "0")) / 1_000_000

        available = session["limit"] - session["reserved"] - session["spent"]
        if amount > available:
            return {"status": "DECLINED", "errorMessage": f"spend limit exceeded: "
                    f"requested ${amount:.4f}, available ${available:.4f}"}

        # Phase 1: reserve
        session["reserved"] += amount
        # Phase 2: "process" (in production: sign + broadcast the on-chain transaction)
        # Phase 3: commit
        session["reserved"] -= amount
        session["spent"] += amount

        return {
            "status": "COMPLETED",
            "amountPaid": {"value": f"{amount:.4f}", "currency": "USDC"},
            "remainingBudget": {"value": f"{session['limit'] - session['spent']:.4f}", "currency": session["currency"]},
            "paymentProof": f"0xsim{uuid.uuid4().hex[:40]}",
            "resource": x402.get("resource", ""),
        }

print("✅ Simulated payment SDK ready (interface-identical to bedrock_agentcore.payments)")

## 400.2 · One-time setup, instrument, session, and autonomous payment

This is the exact four-step flow from AWS's documented production pattern: configure the
payment manager and connector once, provision a funded instrument (embedded wallet),
scope a session with a hard spend limit, then let the agent call `process_payment` as
many times as it needs — the budget enforces itself.

In [ ]:
payment_client = SimulatedPaymentClient(region_name=REGION)

# ── One-time setup ──
setup_response = payment_client.create_payment_manager_with_connector(
    payment_manager_name="novapay-compliance-payments",
    payment_manager_description="Pays third-party sanctions/watchlist data vendors on behalf of the compliance agent",
    authorizer_type="AWS_IAM",
    role_arn=f"arn:aws:iam::{ACCOUNT}:role/NovaPay-PaymentRole-{RUN}",
    payment_connector_config={
        "name": "sanctionsDataVendorConnector",
        "description": "Connector for the premium sanctions-screening vendor",
        "payment_credential_provider_config": {
            "name": "novapayVendorCredential",
            "credential_provider_vendor": "COINBASE",
            "credentials": {
                "api_key_id": "SIMULATED_KEY_ID",
                "api_key_secret": "SIMULATED_KEY_SECRET",
                "wallet_secret": "SIMULATED_WALLET_SECRET",
            },
        },
    },
)
payment_manager_arn = setup_response["paymentManager"]["paymentManagerArn"]
payment_connector_id = setup_response["paymentManager"]["paymentConnectorId"]
print(f"✅ Payment manager: {payment_manager_arn}")

manager = SimulatedPaymentManager(payment_manager_arn=payment_manager_arn, region_name=REGION)

# ── Fund an instrument (embedded wallet) ──
instrument = manager.create_payment_instrument(
    user_id="novapay-compliance-agent",
    payment_connector_id=payment_connector_id,
    payment_instrument_type="EMBEDDED_CRYPTO_WALLET",
    payment_instrument_details={
        "embeddedCryptoWallet": {
            "network": "BASE_SEPOLIA",  # public testnet — no real funds, safe for labs
            "linkedAccounts": [{"email": {"emailAddress": "compliance-agent@novapay.example"}}],
        }
    },
)
PAYMENT_INSTRUMENT_ID = instrument["paymentInstrumentId"]
print(f"✅ Instrument funded: {PAYMENT_INSTRUMENT_ID}")

# ── Scope a session: the agent's financial boundary ──
session_response = manager.create_payment_session(
    user_id="novapay-compliance-agent",
    limits={"maxSpendAmount": {"value": "10.00", "currency": "USD"}},
    expiry_time_in_minutes=60,
)
PAYMENT_SESSION_ID = session_response["paymentSessionId"]
print(f"✅ Payment session: {PAYMENT_SESSION_ID}  (limit: $10.00, expires in 60 min)")
print(f"   The agent cannot spend beyond this limit or outside this window — enforced")
print(f"   atomically by AgentCore Payments, not by application code.")

In [ ]:
# ── The agent autonomously pays for three corridor lookups ──
# Each call simulates the vendor returning HTTP 402 with an x402 payment payload;
# the agent (via AgentCore Payments) pays and retries — matching the real flow.

lookups = [
    {"corridor": "NG-GH", "resource": "https://sanctions-vendor.example/screen/NG-GH"},
    {"corridor": "NG-UK", "resource": "https://sanctions-vendor.example/screen/NG-UK"},
    {"corridor": "GH-KE", "resource": "https://sanctions-vendor.example/screen/GH-KE"},
]

print(f"Compliance agent researching {len(lookups)} corridors for Growth's expansion analysis:\n")
for lookup in lookups:
    with tracer.start_as_current_span("novapay.vendor_payment") as span:
        span.set_attribute("novapay.corridor", lookup["corridor"])
        span.set_attribute("gen_ai.tool.name", "process_payment")

        payment_response = manager.process_payment(
            user_id="novapay-compliance-agent",
            payment_session_id=PAYMENT_SESSION_ID,
            payment_instrument_id=PAYMENT_INSTRUMENT_ID,
            payment_input={
                "cryptoX402": {
                    "version": "1",
                    "payload": {
                        "scheme": "exact",
                        "network": "base-sepolia",
                        "maxAmountRequired": "20000",  # 0.02 USDC in 6-decimal micro-units
                        "resource": lookup["resource"],
                        "description": f"Sanctions screening — corridor {lookup['corridor']}",
                        "mimeType": "application/json",
                        "payTo": "0xVendorWalletAddress",
                        "maxTimeoutSeconds": 300,
                        "asset": "0xUSDCContractAddress",
                        "extra": {"name": "USDC", "version": "2"},
                    },
                }
            },
        )
        span.set_attribute("novapay.payment_status", payment_response.get("status", "UNKNOWN"))

        status = payment_response.get("status")
        if status == "COMPLETED":
            paid = payment_response["amountPaid"]["value"]
            remaining = payment_response["remainingBudget"]["value"]
            print(f"  ✅ {lookup['corridor']}: paid ${paid} USDC  →  remaining budget ${remaining}")
        else:
            print(f"  ⚠️  {lookup['corridor']}: {status} — {payment_response.get('errorMessage')}")

print(f"\n✅ 3 merchants, 3 payments, one call each — zero manual billing setup.")

## 400.3 · What changes between this simulation and production

Nothing except the import and the credentials. This is the actual code from AWS's
technical deep-dive on AgentCore payments (Vangara, Ansari & Shriyan, AWS ML Blog,
May 2026) — compare it line-by-line against the simulator above:

```python
from bedrock_agentcore.payments import PaymentClient, PaymentManager

# One-time setup — identical shape to SimulatedPaymentClient above
payment_client = PaymentClient(region_name="us-west-2")
response = payment_client.create_payment_manager_with_connector(
    payment_manager_name="myPaymentManager",
    payment_manager_description="myPaymentManager description",
    authorizer_type="AWS_IAM",
    role_arn=ROLE_ARN,
    payment_connector_config={
        "name": "myPaymentConnector",
        "description": "myPaymentConnector description",
        "payment_credential_provider_config": {
            "name": "myCoinbasePaymentCredential",
            "credential_provider_vendor": "<PROVIDER>",       # real: "COINBASE" or "STRIPE"
            "credentials": {
                "api_key_id": API_KEY,                        # real: your CDP/Stripe key
                "api_key_secret": API_KEY_SECRET,
                "wallet_secret": WALLET_SECRET,
            },
        },
    },
)
payment_manager_arn = response["paymentManager"]["paymentManagerArn"]
payment_connector_id = response["paymentManager"]["paymentConnectorId"]

# Instrument — identical shape, except the real response includes a redirectUrl
# your end user visits (Coinbase-hosted WalletHub) to fund the wallet and grant
# signing authority before first use.
manager = PaymentManager(payment_manager_arn=payment_manager_arn, region_name="us-west-2")
instrument = manager.create_payment_instrument(
    user_id="test-user-123",
    payment_connector_id=payment_connector_id,
    payment_instrument_type="EMBEDDED_CRYPTO_WALLET",
    payment_instrument_details={
        "embeddedCryptoWallet": {
            "network": "ETHEREUM",
            "linkedAccounts": [{"email": {"emailAddress": "test@example.com"}}],
        }
    },
)

# Session and process_payment — byte-for-byte identical calls to the simulator above.
session_response = manager.create_payment_session(
    user_id="test-user-123",
    limits={"maxSpendAmount": {"value": "100.00", "currency": "USD"}},
    expiry_time_in_minutes=60,
)
payment_response = manager.process_payment(
    user_id="user-123",
    payment_session_id=PAYMENT_SESSION_ID,
    payment_instrument_id=PAYMENT_INSTRUMENT_ID,
    payment_input={"cryptoX402": {"version": "1", "payload": {...}}},
)
```

**Source:** [Technical deep dive: AgentCore payments and innovation in agentic commerce](https://aws.amazon.com/blogs/machine-learning/technical-deep-dive-agentcore-payments-and-innovation-in-agentic-commerce/) — AWS Machine Learning Blog, May 2026.

## 400.4 · Production observability and guardrails (reference)

AgentCore Payments emits a three-pillar observability system automatically — metrics
(success/failure counts and latency per operation), structured logs (correlated by
request ID), and distributed traces (W3C trace-context spans enriched with spend amount
and remaining budget) — with **zero instrumentation code required**, the same
zero-code-required principle Lab 9 spent an entire lab manually building for the fraud
gateway. That is a deliberate design choice by AWS: payments infrastructure is
sufficiently high-stakes that observability is not optional or bolted on later.

The concurrency guarantee matters here too: if the compliance agent and a second Growth
agent both draw from the same session simultaneously (booking three data lookups in
parallel, say), AgentCore Payments' reserve → process → commit-or-rollback protocol
guarantees neither reads a stale balance. Whether it's one agent or a thousand
transacting against the same budget, there are no stale reads and no overspending — the
same atomicity NovaPay would otherwise have to hand-build with database locking.

> ### Level 400 checkpoint
> The compliance agent paid three vendors, $0.02 each, with zero manual billing setup,
> a hard $10 session budget it could not exceed, and full observability it didn't have
> to build. That closes NovaPay's Act III.
>
> Put together, Levels 200–400 turned three separate maturity gaps — *nobody can find
> what exists*, *nobody can prove what's good*, *nothing can pay for what it needs* —
> into one governed pipeline: **discover → prove → transact.**

## 400.5 · Teardown

Delete every real AWS resource created in this lab. The payment simulation is in-memory
only and needs no cleanup.

**Note on registry records:** unlike the old `bedrock-agentcore` namespace (which had no
delete API for individual records — `DEPRECATED` was the only terminal cleanup state),
the current `agent-registry` namespace added a real `DeleteRegistryRecord` operation. We
use it directly, then delete the registry itself; if the registry still refuses deletion
with records attached, the `safe()` wrapper below reports it clearly rather than failing
the whole cell.

In [ ]:
errors = []

def safe(fn, label):
    try:
        fn()
        print(f"✅ {label}")
    except Exception as e:
        errors.append((label, str(e)))
        print(f"⚠️  {label}: {e}")

# 1. Delete the registry record
if RECORD_ID:
    safe(lambda: registry_control.delete_registry_record(
             registryId=REGISTRY_ID, recordId=RECORD_ID),
         "Delete registry record")
    time.sleep(3)

# 2. Registry
if REGISTRY_ID:
    safe(lambda: registry_control.delete_registry(registryId=REGISTRY_ID),
         "Delete registry")

# 3. OTEL cleanup
provider.shutdown()
print("✅ OpenTelemetry provider shut down")

print(f"\n{'='*60}")
if errors:
    print(f"⚠️  {len(errors)} resource(s) may need manual cleanup")
else:
    print(f"✅ All resources deleted")

---
# Knowledge Check

## Concepts

1. **Why does this lab build Registry, then Evaluations, then Payments — not the reverse?**
   Each stage is a governance gate for the next: you cannot evaluate what you cannot
   find (Registry before Evaluations), and you should not give financial autonomy to an
   agent nobody has scored (Evaluations before Payments).

2. **What is the difference between a registry record's `DRAFT`, `PENDING_APPROVAL`, and
   `APPROVED` states?**
   `DRAFT` is created but not submitted. `PENDING_APPROVAL` has been submitted and
   awaits a curator (unless auto-approval is on). `APPROVED` is discoverable via search.
   Only `APPROVED` records appear in search results.

3. **What is the difference between on-demand and online evaluation?**
   On-demand scores a specific session you choose, on request (a go/no-go decision or CI
   gate). Online continuously samples live production traffic against configured
   evaluators (an ongoing quality dashboard).

4. **What does the `evaluationTarget` parameter let you do?**
   Score a specific trace or span within a session, rather than the whole session — the
   pattern a CI pipeline would use to gate one regression test.

5. **What problem does AgentCore Payments' three-phase (reserve → process →
   commit/rollback) protocol solve?**
   Concurrent overspending. Without atomic reservation, two simultaneous payments from
   the same session could both read a stale balance and together exceed the budget.

## Architecture

6. **Why does AWS Agent Registry validate MCP/A2A records against real protocol schemas
   at submission time?**
   So a broken or malformed record is rejected immediately, rather than discovered later
   by a confused consumer who assumed anything in the registry is guaranteed valid.

7. **What is the x402 protocol's role in AgentCore Payments?**
   It is the machine-to-machine payment negotiation protocol: a paid endpoint returns
   HTTP 402 with payment requirements, the payer generates a cryptographic payment
   proof, and the endpoint is retried with that proof attached.

8. **Why is a payment session scoped with both a spend limit and an expiry time?**
   The spend limit bounds how much an agent can commit; the expiry bounds how long that
   authority lasts. Together they make the agent's financial boundary both amount-safe
   and time-safe — an agent cannot extend its own session.

## Operations

9. **What changed between this lab's payment simulator and the real
   `bedrock_agentcore.payments` SDK?**
   Only the import statement and the credentials passed into
   `create_payment_manager_with_connector`. Every method name, parameter name, and
   response shape is identical.

10. **What observability does AgentCore Payments provide without any instrumentation
    code?**
    Metrics (success/failure/latency per operation), structured logs correlated by
    request ID, and W3C-compatible distributed traces enriched with spend amount and
    remaining budget — delivered automatically to CloudWatch.

## Summary

| What you built | Why it matters |
|---|---|
| AWS Agent Registry with an approved, searchable record | Other teams find NovaPay's existing agents and tools instead of rebuilding them |
| An explicit submit → approve → search governance flow | Discovery without a curation gate is noise at scale |
| A real Evaluate() call against real OTEL-derived spans | Growth's adoption decision is backed by a score, not an assurance |
| Session- and trace-level evaluation targets | The same API scales from "score this whole session" to "gate this one regression" |
| An interface-identical payment simulator, with the real code shown side-by-side | Hands-on practice with the exact production API, with zero third-party account required |
| A budget-scoped, atomically-enforced autonomous payment flow | Agents can now transact — safely, auditably, and within a hard financial boundary |

**The arc across Labs 7–10:** Memory gave agents recall. Gateway gave them governed
access. Observability gave them accountability. Registry, Evaluations, and Payments give
them an economy — discoverable, trusted, and, when earned, empowered to act with real
consequences.